In [ ]:
import numpy as np
import time, random, logging

from genetic_operators import mutate, crossover
from schema import Generation, Network
from fitness import fitness_score
from config import FITNESS_WEIGHTS

from utils.time_utils import format_elapsed, ElapsedTimeFormatter
from utils.checkpoint_utils import load_ga_checkpoint, save_recent_best_population, save_ga_checkpoint, make_run_tag

In [ ]:
# Logger object. Use debug, info, warning, error accordingly
logger = logging.getLogger("genetic_algorithm")
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    handler = logging.StreamHandler()

    formatter = ElapsedTimeFormatter("[%(elapsed)s] %(levelname)s: %(message)s")

    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False

In [ ]:
# GA VERSION 2 - Finding the optimal Network

# This implementation takes only 2 parents from the whole generation and generates the population from them
def perform_genetic_algorithm(
    network_population,
    graph_of_stops,
    total_potential_transport_nodes,
    weights,
    mutation_probability,
    num_mutations_probabilities,
    plateau_threshold,
    plateau_count_limit,
    max_gens,
    run_tag="",
    checkpoint=None,
    checkpoint_every=1,
    debug=False
):
    # Sets logging to debug mode meaning logger.debug() works
    logger.setLevel(logging.DEBUG if debug else logging.INFO)

    max_fitness_score = 0

    if checkpoint is not None:
        logger.info("Checkpoint found. Resuming previous GA run...")
        generation_num = checkpoint["generation_num"]
        network_population = checkpoint["population"]
        max_score_list = checkpoint["max_score_list"]
        most_optimal_gen = checkpoint["most_optimal_gen"]
        prev_max_fitness_score = checkpoint["prev_max_fitness_score"]
        plateau_counter = checkpoint["plateau_counter"]
        current_highest_max_score = checkpoint["current_highest_max_score"]
        current_highest_avg_score = checkpoint["current_highest_avg_score"]
        elapsed_seconds_before = checkpoint.get("elapsed_seconds", 0.0)

        if "python_random_state" in checkpoint:
            random.setstate(checkpoint["python_random_state"])
        if "numpy_random_state" in checkpoint:
            np.random.set_state(checkpoint["numpy_random_state"])

        logger.info(f"Resuming from generation {generation_num}")
        logger.info(f"Accumulated runtime so far: {format_elapsed(elapsed_seconds_before)}")
    else:
        max_score_list = []
        prev_max_fitness_score = None
        plateau_counter = 0
        most_optimal_gen = None
        current_highest_max_score = 0
        current_highest_avg_score = 0
        generation_num = 1
        elapsed_seconds_before = 0.0
        logger.info("No checkpoint found. Starting fresh GA run...")

    session_start_time = time.time()

    try:
        while True:
            logger.debug(f"Generation {generation_num}")

            # Evaluate the fitness of each network in the population
            for network in network_population:
                network_graph = network.graph
                if network.fitness_score == 0:
                    network.fitness_score, components = fitness_score(network_graph, weights, total_potential_transport_nodes)
                    setattr(network, 'fitness_score_components', components)

            # Sort the network population by fitness score
            sorted_network_population = sorted(network_population, key=lambda x: x.fitness_score, reverse=True)
            fitness_scores = [network.fitness_score for network in sorted_network_population]

            # For printing purposes in order
            for network in sorted_network_population:
                logger.debug(f"Network Score: {network.fitness_score}")

            # The max network score of this generation
            max_fitness_score = sorted_network_population[0].fitness_score
            max_score_list.append([max_fitness_score, sorted_network_population[0].fitness_score_components])

            # The average network score of this generation
            total_score = sum(fitness_scores)
            average_score = total_score / len(fitness_scores)

            logger.info(f"Generation {generation_num} Max Score: {max_fitness_score} Average Score: {average_score}")

            # Check if the score is optimal. If it is, then stop
            # if max_fitness_score >= optimal_fitness_score:
            #     print(f"WE FOUND THE OPTIMAL NETWORK IN GENERATION {generation_num}")
            #     most_optimal_network = sorted_network_population[0]
            #     print(f"Fitness Score of the most optimal network: {most_optimal_network.fitness_score}")
            #     break

            # Check if this generation has the highest max fitness score and highest average score
            if max_fitness_score >= current_highest_max_score and average_score >= current_highest_avg_score:
                current_highest_avg_score = average_score
                current_highest_max_score = max_fitness_score
                most_optimal_gen = sorted_network_population
                save_recent_best_population(generation_num, most_optimal_gen, run_tag=f"_{run_tag}")

            # Check if score plateaus based on max score
            if prev_max_fitness_score is not None and abs(max_fitness_score - prev_max_fitness_score) <= plateau_threshold:
                plateau_counter += 1
                if plateau_counter >= plateau_count_limit:
                    logger.info(f"Plateau detected in generation {generation_num}. Stopping...")
                    break
            else:
                plateau_counter = 0
            prev_max_fitness_score = max_fitness_score

            # Save resumable checkpoint every N generations
            if generation_num % checkpoint_every == 0:
                elapsed_now = elapsed_seconds_before + (time.time() - session_start_time)
                save_ga_checkpoint(
                    run_tag=f"_{run_tag}",
                    generation_num=generation_num,
                    network_population=network_population,
                    max_score_list=max_score_list,
                    most_optimal_gen=most_optimal_gen,
                    prev_max_fitness_score=prev_max_fitness_score,
                    plateau_counter=plateau_counter,
                    current_highest_max_score=current_highest_max_score,
                    current_highest_avg_score=current_highest_avg_score,
                    elapsed_seconds=elapsed_now,
                )
                # logger.info(f"Checkpoint saved at generation {generation_num} | elapsed: {format_elapsed(elapsed_now)}")

            # Choosing 10% of the networks to be parents using Roulette Wheel Selection
            logger.debug("Choosing parents...")

            #Getting the number of parents to be selected
            num_parents = int(len(network_population) * 0.10)
            if num_parents % 2 == 1:
                num_parents += 1

            # List of parents
            parent_networks = []
            logger.debug(f"Number of parents selected: {num_parents}")
            for i in range(num_parents):
                # Get the list of all fitness scores
                fitness_scores = [network.fitness_score for network in sorted_network_population]

                # Shift the scores to ensure all are positive
                min_score = min(fitness_scores)
                if min_score < 0:
                    fitness_scores = [score - min_score for score in fitness_scores]

                # Get the probabilities
                total = sum(fitness_scores)
                selection_p = [score / total for score in fitness_scores]

                # Getting the parent
                chosen_parent = np.random.choice(sorted_network_population, 1, p=selection_p)[0]
                logger.debug(f"Parent {i} score: {chosen_parent.fitness_score}")

                sorted_network_population.remove(chosen_parent)
                parent_networks.append(chosen_parent)


            # Add back all the removed parent networks
            sorted_network_population.extend(parent_networks)

            # Sort the networks by fitness function again
            sorted_network_population = sorted(network_population, key=lambda x: x.fitness_score, reverse=True)

            # Pairing the parents
            parent_pairs = [parent_networks[i:i+2] for i in range(0, len(parent_networks), 2)]

            # Each parent pair will now produce children
            logger.debug("Generating children through crossover...")
            children_networks = []
            for pair in parent_pairs:
                parent1 = pair[0]
                parent2 = pair[1]

                # Get 2 children from crossovers between the two parents
                child1, child2, is_different = crossover(parent1, parent2)

                # Getting the number of mutations. If the children are entirely different from their parents, there should be atleast one mutation
                if is_different:
                    index_array = list(range(len(num_mutations_probabilities)))
                    num_mutations = np.random.choice(index_array, 1, p=num_mutations_probabilities)[0]
                else:
                    temp_mutation_probs = [0, 0.1, 0.4, 0.3, 0.2]
                    index_array = list(range(len(temp_mutation_probs)))
                    num_mutations = np.random.choice(index_array, 1, p=temp_mutation_probs)[0]

                # # Getting the number of mutations.
                # index_array = list(range(len(num_mutations_probabilities)))
                # num_mutations = np.random.choice(index_array, 1, p=num_mutations_probabilities)[0]

                for _ in range(num_mutations):
                    # Apply mutations to the children based on mutation probability hyperparameter
                    if np.random.rand() < mutation_probability:
                        mutate(child1, graph_of_stops)

                    if np.random.rand() < mutation_probability:
                        mutate(child2, graph_of_stops)

                # Add the children to the list of children
                children_networks.append(child1)
                children_networks.append(child2)


            # Preparing the next generation

            # Remove the lowest scored networks and replace it with the children
            network_population = sorted_network_population[:-len(children_networks)]
            network_population.extend(children_networks)

            logger.debug(f"Completed generation {generation_num}. Proceeding to next generation...")

            # Increment the generation number
            generation_num += 1
            if generation_num > max_gens:
                logger.info(f"Reached maximum generation count of {generation_num}. Stopping...")
                break

    except KeyboardInterrupt:
        elapsed_now = elapsed_seconds_before + (time.time() - session_start_time)
        save_ga_checkpoint(
            run_tag=f"_{run_tag}",
            generation_num=generation_num,
            network_population=network_population,
            max_score_list=max_score_list,
            most_optimal_gen=most_optimal_gen,
            prev_max_fitness_score=prev_max_fitness_score,
            plateau_counter=plateau_counter,
            current_highest_max_score=current_highest_max_score,
            current_highest_avg_score=current_highest_avg_score,
            elapsed_seconds=elapsed_now,
        )

        logger.info(f"GA interrupted. Checkpoint saved at generation {generation_num}...")
        logger.info(f"Saved elapsed runtime: {format_elapsed(elapsed_now)}")
        raise

    # Sorts before returning
    if most_optimal_gen is not None:
        most_optimal_gen = sorted(most_optimal_gen, key=lambda x: x.fitness_score, reverse=True)

    total_elapsed = elapsed_seconds_before + (time.time() - session_start_time)

    logger.info(f"Highest score of all generations: {current_highest_max_score}")
    logger.info(f"Total GA runtime: {format_elapsed(total_elapsed)}")

    save_ga_checkpoint(
        run_tag=f"_{run_tag}",
        generation_num=generation_num,
        network_population=network_population,
        max_score_list=max_score_list,
        most_optimal_gen=most_optimal_gen,
        prev_max_fitness_score=prev_max_fitness_score,
        plateau_counter=plateau_counter,
        current_highest_max_score=current_highest_max_score,
        current_highest_avg_score=current_highest_avg_score,
        elapsed_seconds=total_elapsed,
    )

    return most_optimal_gen, max_score_list, total_elapsed

In [ ]:
mutation_probability = 0.2
num_mutations_probabilities = [0.1, 0.1, 0.4, 0.2, 0.2]
plateau_threshold = 0.00000000001
plateau_max_count = 10000
max_generations = 50000

run_tag = make_run_tag(FITNESS_WEIGHTS)

resume = False # Set to true if u wanna use a checkpoint
checkpoint = (load_ga_checkpoint(run_tag=f"_{run_tag}") if resume else None)

perform_genetic_algorithm([], [], [], FITNESS_WEIGHTS, mutation_probability, num_mutations_probabilities,
                          plateau_threshold, plateau_max_count, max_generations, run_tag, checkpoint)